# Lesson 02 — 教师演练：From MetaQA to Knowledge Graph

这是教师参考分支的可运行演练，不是学生起步分支的实现。目标是读懂真实数据、构建保留关系的图、验证有界简单路径。

按顺序执行：配置 → 数据 → 邻接表 → 小图路径 → 真实两跳问题 → 测试与复盘。本节使用CPU与Python标准库，不训练神经网络。

运行前按本分支README准备代码与数据。数据不会随Git clone下载；当前本机数据在src/dataset。若修改src中的函数，重启内核后从头运行，避免旧import缓存。

## 0–20 min：目标、环境与仓库

先口述：今天的输出为什么是图路径，而不是训练成绩？确认自己在lesson02-teacher-rehearsal分支。

在VS Code打开本Notebook，从项目根或notebooks目录运行。Colab先clone本分支并切换到项目根，数据路径另行准备。METAQA_DATA_DIR可覆盖默认数据目录。

In [ ]:
from pathlib import Path
import os
import sys
import platform

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "load_metaqa.py").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("请先进入教师分支项目目录")
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = Path(os.environ.get("METAQA_DATA_DIR", str(PROJECT_ROOT / "src" / "dataset")))
HOP = 2
VARIANT = "vanilla"
N_QUESTIONS = 3
MAX_HOPS = 2
MAX_PATHS = 20
print("Project:", PROJECT_ROOT)
print("Python:", platform.python_version())
print("Dataset:", DATA_DIR)


## 20–40 min：从原始行到样本

先解释三种分隔：三元组的竖线、QA字段的TAB、多答案的竖线。方括号给出主题实体。

打开src/load_metaqa.py；说清parse_question的输入输出，再运行。问题ID与split保留下来，避免下一课将同题候选拆入不同集合。

In [ ]:
kb_path = DATA_DIR / "kb.txt"
qa_path = DATA_DIR / f"{HOP}-hop" / VARIANT / "qa_train.txt"
for path in (kb_path, qa_path):
    if not path.is_file():
        raise FileNotFoundError(f"数据未准备：{path}；请按README下载或设置METAQA_DATA_DIR")
with kb_path.open(encoding="utf-8") as stream:
    print("KG原始行:", next(stream).rstrip())
with qa_path.open(encoding="utf-8") as stream:
    print("QA原始行:", repr(next(stream).rstrip()))


In [ ]:
from collections import Counter
from src.load_metaqa import load_triples, load_questions, parse_question

triples = load_triples(kb_path)
questions = load_questions(qa_path, split="train", limit=N_QUESTIONS)
print("真实KG三元组数:", len(triples))
print("关系统计:", dict(Counter(relation for _, relation, _ in triples)))
for sample in questions:
    print(sample["question_id"], sample["question"])
    print("  topic:", sample["topic_entity"], "answers:", sorted(sample["answers"]))


### 预测后再运行：多答案

下面是单独的格式练习，不是声称来自原始MetaQA记录。预测返回的answers有几个元素；为什么它不能始终是一个字符串？

In [ ]:
format_example = parse_question("who acted in [FilmX]?\tActor A|Actor B", "example:1", "example")
print(format_example)
assert format_example["answers"] == {"Actor A", "Actor B"}


## 40–65 min：三元组到邻接表

打开src/build_graph.py。每条边保存邻居、关系、inverse标记和原始三元组。同一对实体的导演/编剧关系都要保留。

先预测：原事实FilmX directed_by PersonA，反向访问时原始triple是否应改写？运行后检查inverse字段。

In [ ]:
from src.build_graph import build_graph

graph = build_graph(triples)
source = questions[0]["topic_entity"]
print("图节点数:", len(graph))
print("主题实体:", source)
print("前五条邻接记录:")
for edge in graph.get(source, [])[:5]:
    print(edge)

parallel_graph = build_graph([("FilmX", "directed_by", "PersonA"),
                              ("FilmX", "written_by", "PersonA")])
print("同端点两种关系:", parallel_graph["FilmX"])
print("反向访问:", parallel_graph["PersonA"])


## 65–70 min：休息与口头检查

合上参考代码，说出一个邻接记录的四个字段。请另一位同学检查反向访问和原始事实的区别。

## 70–95 min：先在小图上验证路径

打开src/path_retrieval.py。先手画下方小图，预测A到D最多两跳有哪些路径，再运行。

Simple path不重复访问节点；visited只属于当前路径。平行关系可以使相同节点序列对应不同证据。

In [ ]:
from src.path_retrieval import find_paths, path_nodes, format_path

toy_triples = [("A", "r", "B"), ("A", "s", "C"),
               ("B", "t", "D"), ("C", "u", "D"),
               ("B", "v", "C"), ("A", "alternate", "B")]
toy_graph = build_graph(toy_triples)
toy_paths = find_paths(toy_graph, "A", "D", max_hops=MAX_HOPS)
for path in toy_paths:
    print(format_path("A", path))
    nodes = path_nodes("A", path)
    assert len(nodes) == len(set(nodes))
    assert len(path) <= MAX_HOPS
print("不同关系路径数:", len(toy_paths))


### 小练习：改变边界

先预测max_hops=1的结果。再试max_hops=3，解释新路径从哪里来；不要直接在整个真实图上放开搜索深度。

In [ ]:
print("最多1跳:", find_paths(toy_graph, "A", "D", max_hops=1))
print("不存在的目标:", find_paths(toy_graph, "A", "missing", max_hops=2))
print("最多3跳:", len(find_paths(toy_graph, "A", "D", max_hops=3)))


### 再回到真实MetaQA问题

下面使用已知Train答案作为目标，验证路径函数及事实方向。这是检索检查，不是未知问题的候选生成，更不是QA准确率。

下一课再实现独立于未知题gold的候选规则。本次最多展示少量训练问题与每题MAX_PATHS条路径，截断设置要记录。

In [ ]:
checks = []
for sample in questions:
    start = sample["topic_entity"]
    target = sorted(sample["answers"])[0]
    paths = find_paths(graph, start, target, max_hops=MAX_HOPS, max_paths=MAX_PATHS)
    print("\nQuestion:", sample["question"])
    print("检查目标:", target, "返回路径数:", len(paths), "输出上限:", MAX_PATHS)
    for path in paths[:3]:
        print(" ", format_path(start, path))
        nodes = path_nodes(start, path)
        assert len(nodes) == len(set(nodes))
        assert nodes[-1] == target
        assert len(path) <= MAX_HOPS
    checks.append({"question_id": sample["question_id"], "target": target,
                   "paths_returned": len(paths), "max_paths": MAX_PATHS})
print("\n检索检查记录:", checks)


## 95–110 min：预期、测试与解释

先说出应该覆盖的情况：多答案解析、坏行、同端点不同关系、反向访问、菱形多路径、环、不可达、跳数边界与输出上限。

tests/test_lesson02.py只用小图，不需要数据文件。测试通过后，仍需解释真实路径中的每条关系。

In [ ]:
import subprocess

result = subprocess.run([sys.executable, "-s", "-X", "utf8", "-m", "unittest", "discover", "-s", "tests", "-v"],
                        cwd=PROJECT_ROOT, capture_output=True, text=True, encoding="utf-8")
print(result.stdout)
print(result.stderr)
result.check_returncode()


## 110–120 min：复盘与提交

检查点：能解析真实QA与三元组；能解释邻接记录和inverse；能检索并核验有界简单路径。模型训练、PHL生成和完整候选算法留待后续。

模拟从零操作时，从lesson02-student-start另建自己的练习分支；教师参考放在另一个窗口或GitHub页面逐函数对照。学生分支不合并教师答案。

提交前运行git status，确认只有源码、Notebook和测试，src/dataset仍被忽略。用自己的话记录一个修复与一个尚不确定的问题。